# 14 - Louvain split, validation-selected protocol

Same split, arms and budget as notebook 13 (3 Louvain hospitals, 5 seeds, rounds=8, local_epochs=4,
local_only_epochs=32, hidden=128). **What changes is how the reported number is chosen:**

- Every round each hospital scores the global model on its **validation and test** rows at 29 cutoffs
  (0.150 ... 0.850). The server picks the **(round, cutoff) with the best size-weighted validation
  macro-F1**; the test score is read **once, at that choice**. Nothing is picked on test any more.
- Local-only gets the same rule: scored every 4 epochs (the federated cadence), (epoch, cutoff)
  chosen per hospital on validation, test read there.
- The old numbers (best round on test, fixed 0.5) are still computed and stored (`fedavg_best`,
  `local_only`, ...), so this run is directly comparable with `amr_louvain_full.json`.
- New keys are prefixed `val_` (`val_fed`, `val_fed_at05`, `val_fed_final`, `val_worst_fed`,
  `val_local`, ...). See `src/amr_fed/federated/selection.py`.

Runs on **Colab (GPU)** or **locally**; the first cell detects which. Results go to
`results/amr_louvain_val.json` (checkpointed after every arm, resumes if present; on Colab it is also
mirrored to Drive so a disconnect does not lose finished arms).

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/federated-baselines'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + imports (the selection module must be present = right branch)
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import selection            # fails here if the branch is stale
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split
from amr_fed.topology import compute_fingerprint, aggregation_weights
print('imports OK | cutoffs:', selection.THRESHOLDS[[0, selection.IDX_05, -1]])

In [ ]:
# 3) Load cohort once + smoke (expect 3 hospitals per seed; 42/46 and 43/45 share a partition)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
for s in SEEDS:
    a = louvain_split(df, n_clients=3, seed=s)
    print(f'seed {s}: {a.nunique()} hospitals | patients (largest first): {a.value_counts().sort_values(ascending=False).tolist()}')
    assert a.index.is_unique and len(a) == df[PK].nunique(), 'not 1:1'
print('SMOKE PASS')

In [ ]:
# 4) All arms, identical protocol. Resumes from results/amr_louvain_val.json (and Drive mirror on Colab).
import os, ray, json, shutil
from pathlib import Path
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: louvain_split(d, n_clients=3, seed=s),
          n_clients=3, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
ARMS = [  # (key, label, extra kwargs)
    ('fedavg',        'louvain/FedAvg',        dict(strategy='fedavg')),
    ('fedprox',       'louvain/FedProx',       dict(strategy='fedprox')),
    ('fedmedian',     'louvain/FedMedian',     dict(strategy='fedmedian')),
    ('fedtrimmedavg', 'louvain/FedTrimmedAvg', dict(strategy='fedtrimmedavg')),  # == uniform at 3 hospitals (trims 0)
    ('uniform',       'louvain/UNIFORM (T=inf control)', dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=float('inf'))),
    ('topo',          'louvain/TOPO (T=1)',    dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0)),
]
RESULTS = SRC.parent / 'results' / 'amr_louvain_val.json'
RESULTS.parent.mkdir(exist_ok=True)
MIRROR = Path('/content/drive/MyDrive/amr_louvain_val.json') if ON_COLAB else None
src = RESULTS if RESULTS.exists() else (MIRROR if MIRROR and MIRROR.exists() else None)
res = json.load(open(src)) if src else {}
for i, (key, label, extra) in enumerate(ARMS):
    if key in res:
        print(f'skip {key}: already saved')
        continue
    ray.shutdown()
    res[key] = run_multiseed(label=label, compute_pooled=(i == 0), **KW, **extra)
    json.dump(res, open(RESULTS, 'w'), default=str)  # checkpoint after every arm
    if MIRROR: shutil.copy(RESULTS, MIRROR)
    print(f'checkpoint saved after {key}')

In [ ]:
# 5) Summary under the validation-selected protocol (mean +/- std over seeds, test read at the val choice)
import numpy as np
names = {'fedavg':'FedAvg','fedprox':'FedProx','fedmedian':'FedMedian','fedtrimmedavg':'FedTrimmedAvg','uniform':'Uniform','topo':'Topo'}
keys = [k for k, _, _ in ARMS]
def col(k, key): return np.array([r[key] for r in res[k]['runs']], dtype=float)
def f(x): return f'{np.nanmean(x):.4f} +/- {np.nanstd(x):.4f}'
lo = col('fedavg', 'val_local')
print(f"{'method':<14}{'val-sel F1':<18}{'@0.5, val round':<18}{'final, val cutoff':<19}{'worst hospital':<18}{'AUROC':<18}{'val-sel round (per seed)'}")
print(f"{'local-only':<14}{f(lo):<18}{'-':<18}{'-':<19}{f(col('fedavg','val_worst_local')):<18}{f(col('fedavg','val_local_auroc')):<18}")
for k in keys:
    rounds = [r['val_fed_round'] for r in res[k]['runs']]
    print(f"{names[k]:<14}{f(col(k,'val_fed')):<18}{f(col(k,'val_fed_at05')):<18}{f(col(k,'val_fed_final')):<19}{f(col(k,'val_worst_fed')):<18}{f(col(k,'val_fed_auroc')):<18}{rounds}")
print('\nfederated - local-only (val-selected, paired by seed):')
for k in keys:
    d = col(k, 'val_fed') - lo
    print(f'  {names[k]:<14}{d.mean():+.4f}  per seed {np.round(d, 4)}')
print('\nfor reference, the OLD protocol (best round on TEST, fixed 0.5) on this same run:')
print(f"  {'local-only':<14}{f(col('fedavg','local_only'))}")
for k in keys: print(f"  {names[k]:<14}{f(col(k,'fedavg_best'))}  (final {f(col(k,'fedavg_final'))})")

In [ ]:
# 6) Is topology different from the uniform control? Paired by seed (same partition + init per seed)
from scipy import stats
for metric, label in [('val_fed', 'val-selected test F1'), ('val_fed_final', 'final round, val cutoff'), ('val_worst_fed', 'worst hospital')]:
    for other in ('uniform', 'fedmedian', 'fedavg'):
        d = col('topo', metric) - col(other, metric)
        se = d.std(ddof=1) / np.sqrt(len(d))
        lo_ci, hi_ci = d.mean() - stats.t.ppf(.975, len(d)-1) * se, d.mean() + stats.t.ppf(.975, len(d)-1) * se
        print(f'{label:<26} topo - {other:<10} {d.mean():+.4f}  95% CI [{lo_ci:+.4f}, {hi_ci:+.4f}]  p={stats.ttest_1samp(d, 0).pvalue:.2f}  per seed {np.round(d, 4)}')
    print()

In [ ]:
# 7) Per-hospital view, hospitals ranked by size within each seed (val-selected test F1)
ref = res['fedavg']['runs']
def ranked(i):
    sizes = ref[i]['sizes']
    return sorted(range(len(sizes)), key=lambda c: -sizes[c])
print(f"{'hospital':<10}{'patients':>9}{'local':>8}" + ''.join(f'{names[k]:>14}' for k in keys))
for rank, tag in enumerate(['largest', 'middle', 'smallest']):
    n = np.mean([ref[i]['sizes'][ranked(i)[rank]] for i in range(len(SEEDS))])
    l = np.mean([ref[i]['val_local_per_client'][ranked(i)[rank]] for i in range(len(SEEDS))])
    row = [np.mean([res[k]['runs'][i]['val_fed_per_client'][ranked(i)[rank]] for i in range(len(SEEDS))]) for k in keys]
    print(f'{tag:<10}{n:>9.0f}{l:>8.4f}' + ''.join(f'{v:>14.4f}' for v in row))